# **Advanced OOP Concepts**

**DS212 — Object-Oriented Programming**

**Week 12:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Use decorators, class methods, static methods, and metaclasses in appropriate advanced OOP contexts.

This lesson explores features that change **how functions, methods, and classes themselves are built**. They all rest on one idea we have met many times:

> **In Python, functions and classes are objects too.** They can be stored in variables, passed to other functions, returned from functions — and modified.

```{mermaid}
flowchart LR
    F["Functions are objects"] --> D["Decorators<br/>wrap functions & classes"]
    D --> M["@classmethod / @staticmethod<br/>different kinds of methods"]
    C["Classes are objects"] --> MC["Metaclasses<br/>control class creation"]
```

## **Functions Are Objects**

In [ ]:
def shout(text):
    return text.upper() + "!"


yell = shout                         # assign a function to another name
print(yell("hello"))
print(type(shout), shout.__name__)


def apply_twice(func, value):        # pass a function as an argument
    return func(func(value))


print(apply_twice(lambda x: x * 2, 5))


def make_multiplier(factor):         # return a function from a function
    def multiply(x):
        return x * factor            # 'factor' is remembered: a closure
    return multiply


triple = make_multiplier(3)
print(triple(7))

The last example creates a **closure**: the inner function remembers `factor` even after `make_multiplier()` has finished. Closures are the building block of decorators.

## **Function Decorators**

A **decorator** is a function that **takes a function and returns a new function** that usually adds behavior before or after calling the original.

```text
@decorator                 is shorthand for          def func(...):
def func(...):                                           ...
    ...                                              func = decorator(func)
```

```{mermaid}
flowchart LR
    C["caller"] --> W["wrapper()<br/>extra behavior before"]
    W --> O["original function"]
    O --> W2["wrapper()<br/>extra behavior after"]
    W2 --> C
```

In [ ]:
import functools
import time


def timed(func):
    @functools.wraps(func)                # keep the original name and docstring
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f} s")
        return result
    return wrapper


@timed
def slow_sum(n):
    '''Add the numbers from 0 to n-1 using a loop.'''
    total = 0
    for i in range(n):
        total += i
    return total


print(slow_sum(500_000))
print(slow_sum.__name__, "-", slow_sum.__doc__)

`*args, **kwargs` lets the wrapper accept **any** arguments and pass them through. `functools.wraps` copies the original function's name and docstring onto the wrapper, which keeps debugging and documentation accurate.

### **Decorators with arguments**

To configure a decorator, add one more layer: a function that **returns** a decorator.

In [ ]:
def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as error:
                    print(f"  attempt {attempt} failed: {error}")
            raise ConnectionError(f"{func.__name__} failed after {times} attempts")
        return wrapper
    return decorator


calls = {"n": 0}


@retry(times=3)
def fetch_data():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("server busy")
    return "data received"


print(fetch_data())

### **Decorators for validation and logging**

Decorators keep **cross-cutting concerns** — logging, timing, validation, caching, access control — out of the main logic:

In [ ]:
def non_negative(func):
    @functools.wraps(func)
    def wrapper(self, amount):
        if amount < 0:
            raise ValueError(f"{func.__name__}: amount must be non-negative")
        return func(self, amount)
    return wrapper


class Wallet:
    def __init__(self):
        self.balance = 0

    @non_negative
    def deposit(self, amount):
        self.balance += amount

    @non_negative
    def spend(self, amount):
        self.balance -= amount


w = Wallet()
w.deposit(500)
w.spend(120)
print(w.balance)
try:
    w.spend(-50)
except ValueError as error:
    print(error)


@functools.lru_cache(maxsize=None)       # a standard-library decorator: caching
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(80), fib.cache_info().hits)

## **Class Decorators**

Decorators can also be applied to **classes**. A class decorator receives the class object, can inspect or modify it, and returns it (or a replacement). You have already used one: `@dataclass`.

A common use is a **registry** — automatically recording classes so they can be looked up by name:

In [ ]:
EXPORTERS = {}


def register(fmt):
    def decorator(cls):
        EXPORTERS[fmt] = cls             # record the class
        cls.format_name = fmt            # add an attribute to it
        return cls
    return decorator


@register("csv")
class CSVExporter:
    def export(self, rows):
        return "\n".join(",".join(map(str, r)) for r in rows)


@register("md")
class MarkdownExporter:
    def export(self, rows):
        return "\n".join("| " + " | ".join(map(str, r)) + " |" for r in rows)


rows = [["name", "score"], ["Ana", 92]]
for fmt in ("csv", "md"):
    exporter = EXPORTERS[fmt]()
    print(f"[{exporter.format_name}]\n{exporter.export(rows)}")

## **Instance, Class, and Static Methods**

Python's built-in decorators `@classmethod` and `@staticmethod` change **what a method receives** as its first argument.

| Kind               | Decorator        | First parameter      | Can access                         | Typical use                              |
| ------------------ | ---------------- | -------------------- | ---------------------------------- | ---------------------------------------- |
| Instance method    | *(none)*         | `self` — the object  | object **and** class data          | most behavior                            |
| Class method       | `@classmethod`   | `cls` — the class    | class data; can create instances   | alternative constructors, class-level settings |
| Static method      | `@staticmethod`  | *(nothing special)*  | only its arguments                 | utility functions that belong with the class |

```{mermaid}
classDiagram
class Student {
    +int count$
    +String name
    +float gwa
    +__init__(name, gwa)
    +summary() str
    +from_csv_row(row)$ Student
    +from_dict(data)$ Student
    +is_valid_gwa(value)$ bool
}
```

### **Class methods as alternative constructors**

Data arrives in many formats — CSV rows, dictionaries, JSON. Class methods give each format its own **named constructor**:

In [ ]:
class Student:
    count = 0                                   # class attribute

    def __init__(self, name, gwa):
        if not Student.is_valid_gwa(gwa):
            raise ValueError(f"Invalid GWA: {gwa}")
        self.name = name
        self.gwa = gwa
        type(self).count += 1

    def summary(self):                          # instance method
        return f"{self.name}: {self.gwa:.2f}"

    @classmethod
    def from_csv_row(cls, row):                 # alternative constructor
        name, gwa = row.split(",")
        return cls(name.strip(), float(gwa))

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data["gwa"])

    @staticmethod
    def is_valid_gwa(value):                    # utility: no self, no cls
        return 1.0 <= value <= 5.0


students = [
    Student("Ana", 1.25),
    Student.from_csv_row("Ben, 1.75"),
    Student.from_dict({"name": "Carlo", "gwa": 2.0}),
]
for s in students:
    print(s.summary())

print("Created:", Student.count)
print(Student.is_valid_gwa(6.0))

### **Class methods and inheritance**

Because a class method receives `cls`, it creates an instance of **whichever class it was called on** — including subclasses. A static method or a hard-coded `Student(...)` would not:

In [ ]:
class ScholarStudent(Student):
    def summary(self):
        return "★ " + super().summary()


s = ScholarStudent.from_csv_row("Diana, 1.10")
print(type(s).__name__, "|", s.summary())

**Choosing:** if a method needs the **object**, use an instance method. If it needs the **class** (to create objects or use class-level data), use `@classmethod`. If it needs **neither**, but logically belongs with the class, use `@staticmethod` — or consider a plain module-level function.

## **Metaclasses and Controlled Class Creation**

If objects are created from classes... what creates **classes**? Another class, called a **metaclass**. By default, every class is created by `type`.

In [ ]:
class Course:
    pass


print(type(Course()))     # the object's class
print(type(Course))       # the class's class: type
print(type(type))         # type is its own metaclass

# type(name, bases, namespace) can even create a class dynamically
Module = type("Module", (), {"credits": 3, "describe": lambda self: f"{self.credits} units"})
print(Module().describe())

```{mermaid}
flowchart LR
    T["type<br/>(metaclass)"] -->|creates| C["Course<br/>(class)"]
    C -->|creates| O["Course()<br/>(object)"]
```

A **custom metaclass** inherits from `type` and can inspect or modify a class **at the moment the class is defined** — before any object exists. This lets a framework **enforce rules** on every class that uses it.

In [ ]:
class ModelMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if bases:                                       # skip the base class itself
            for required in ("fit", "predict"):
                if not callable(namespace.get(required)):
                    raise TypeError(f"{name} must define {required}()")
            mcls.registry[name.lower()] = cls
        return cls


class Model(metaclass=ModelMeta):
    pass


class MeanModel(Model):
    def fit(self, y):
        self.mean = sum(y) / len(y)
        return self

    def predict(self, n):
        return [self.mean] * n


print(ModelMeta.registry)

try:
    class BrokenModel(Model):          # fails as soon as the class is DEFINED
        def fit(self, y):
            return self
except TypeError as error:
    print("TypeError:", error)

### **A simpler alternative: `__init_subclass__`**

Most metaclass use cases — registering or checking subclasses — can be handled with `__init_subclass__()`, a hook Python calls on the **parent** whenever a subclass is created. It is easier to read and does not conflict with other metaclasses:

In [ ]:
class Plugin:
    plugins = {}

    def __init_subclass__(cls, key=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if not hasattr(cls, "run"):
            raise TypeError(f"{cls.__name__} must define run()")
        Plugin.plugins[key or cls.__name__.lower()] = cls


class Cleaner(Plugin, key="clean"):
    def run(self, data):
        return [x for x in data if x is not None]


class Scaler(Plugin, key="scale"):
    def run(self, data):
        top = max(data)
        return [round(x / top, 2) for x in data]


data = [4, None, 8, 2]
for key in ("clean", "scale"):
    data = Plugin.plugins[key]().run(data)
print(Plugin.plugins.keys(), data)

### **When should you use these features?**

| Feature              | Use when…                                                         | Be careful because…                                  |
| -------------------- | ----------------------------------------------------------------- | ---------------------------------------------------- |
| Function decorator   | the same extra behavior wraps many functions (timing, logging, validation, caching) | too many layers make tracebacks harder to read |
| Class decorator      | you need to register or slightly modify classes                   | the class may look different from its source         |
| `@classmethod`       | you need alternative constructors or class-level behavior         | —                                                    |
| `@staticmethod`      | a helper logically belongs to the class but needs no state        | a module-level function may be simpler               |
| `__init_subclass__`  | subclasses must be registered or checked                          | —                                                    |
| Metaclass            | you are building a **framework** that must control class creation | rarely needed in application code; hard to combine   |

> *"Metaclasses are deeper magic than 99% of users should ever worry about. If you wonder whether you need them, you don't."* — Tim Peters

## **Summary**

| Concept                | Key idea                                                         |
| ---------------------- | ---------------------------------------------------------------- |
| Functions are objects  | they can be passed, returned, and wrapped                        |
| Function decorator     | `@d` replaces `f` with `d(f)`, adding behavior around calls       |
| Class decorator        | `@d` replaces a class with `d(cls)`, e.g., to register it         |
| `@classmethod`         | receives `cls`; ideal for alternative constructors               |
| `@staticmethod`        | receives no implicit argument; a utility grouped with the class  |
| Metaclass              | the class of a class; controls how classes are created           |
| `__init_subclass__`    | a simpler hook for customizing subclass creation                 |

## **References**

Downey, A. B. (2024). *Think Python: How to think like a computer scientist* (3rd ed.). O'Reilly Media. https://allendowney.github.io/ThinkPython/

Lott, S. F., & Phillips, D. (2021). *Python object-oriented programming* (4th ed.). Packt Publishing.

Python Software Foundation. (2026). *functools — Higher-order functions and operations on callable objects*. https://docs.python.org/3/library/functools.html

Python Software Foundation. (2026). *The Python language reference: 3. Data model*. https://docs.python.org/3/reference/datamodel.html

Ramalho, L. (2022). *Fluent Python: Clear, concise, and effective programming* (2nd ed.). O'Reilly Media.

Smith, K. D., Jewett, J. J., Montanaro, S., & Baxter, A. (2003). *PEP 318 — Decorators for functions and methods*. Python Software Foundation. https://peps.python.org/pep-0318/

Talin. (2006). *PEP 3115 — Metaclasses in Python 3000*. Python Software Foundation. https://peps.python.org/pep-3115/

Teichmann, M. (2016). *PEP 487 — Simpler customisation of class creation*. Python Software Foundation. https://peps.python.org/pep-0487/